In [6]:
import os
import json
import gradio as gr
import requests
import sqlite3
from dotenv import load_dotenv
from IPython.display import Markdown, display
from openai import OpenAI

load_dotenv(override=True)
api_key = os.getenv('OPENROUTER_API_KEY')
BASE_URL="https://openrouter.ai/api/v1"
openai = OpenAI(api_key=api_key, base_url=BASE_URL)
MODEL_GPT = 'gpt-4o-mini'
MODEL_LLAMA = 'llama3.2'

In [11]:
system_prompt = "You are an expert technical explainer. Respond with clear, concise explanations. If asked for resources, provide links."

def explain_technical_question(question, model=MODEL_GPT):
    response = openai.chat.completions.create(
        model=model,
        messages=[
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": question}
        ],
    )
    explanation = response.choices[0].message.content
    display(Markdown(explanation))

In [12]:
def web_search_tool(query):
    # Simple DuckDuckGo search API
    url = "https://duckduckgo.com/"
    params = {'q': query}
    resp = requests.post(url, data=params)
    # DuckDuckGo doesn't have a public API, so we return a Google search link instead
    return f"https://www.google.com/search?q={query}"

# Gradio UI

In [18]:
def gradio_response(question, model, web_search_enabled):
    if web_search_enabled:
        search_link = web_search_tool(question)
        search_md = f"\n\n[Web search results]({search_link})"
    else:
        search_md = ""
    # Use explain_technical_question to get the explanation
    response = openai.chat.completions.create(
        model=model,
        messages=[
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": question}
        ],
    )
    explanation = response.choices[0].message.content
    return explanation + search_md

with gr.Blocks() as ui:
    gr.Markdown("# Technical Q&A Assistant")
    question = gr.Textbox(label="Ask a technical question")
    model = gr.Dropdown(choices=[MODEL_GPT, MODEL_LLAMA], value=MODEL_GPT, label="Choose Model")
    web_search_enabled = gr.Checkbox(label="Enable Web Search")
    output = gr.Markdown()
    submit = gr.Button("Submit")

    submit.click(
        fn=gradio_response,
        inputs=[question, model, web_search_enabled],
        outputs=output
    )

ui.launch()

* Running on local URL:  http://127.0.0.1:7862
* To create a public link, set `share=True` in `launch()`.
